# GEAP Agent Evaluation — Interactive Demo Notebook

This notebook walks the **Quality Flywheel** end-to-end with 100% coverage of Google's
[Gemini Enterprise Agent Platform → Optimize → Evaluation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/agent-evaluation) docs.

Each section links the doc page it covers and calls the matching demo step. Where the SDK
returns a rich result, we call `.show()` for interactive tables (a feature of the
[view-results](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/view-results) page).

See also: [`docs/evaluation_demo.md`](../../../docs/evaluation_demo.md) and the coverage
matrix in [`docs/eval_operations.md` §0](../../../docs/eval_operations.md).


## Setup


In [ ]:
import vertexai
from src.config import AGENT_ENGINE_ID
from src.eval.demo import steps

client = steps.make_client()            # Agent Platform SDK client (Vertex)
RESOURCE = steps.resolve_resource(AGENT_ENGINE_ID)
AGENT = 'coordinator_agent'
print('client ready:', client is not None, '| agent resource:', RESOURCE)


## Phase 1 — Design: the Metric Registry
📖 [manage-metrics](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/manage-metrics)

Three metric types: predefined rubric (`types.RubricMetric.*`), custom LLM-as-judge
(`types.LLMMetric`), and custom deterministic code (`types.CodeExecutionMetric`), plus a
reference-based Exact Match. Register once, reuse across offline runs and online monitors.


In [ ]:
from src.eval import metric_registry as mr
print('custom metrics:', [getattr(m, 'name', type(m).__name__) for m in mr.custom_metrics()])
# Register them in the Metric Registry (writes to your project):
# mr.register_all(client)
steps.register_metrics(client)['catalog']


## Phase 2a — Rapid evaluation + `result.show()`
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

A quick pointwise LLM-judge run over a few prompts. `result.show()` renders the aggregate
and per-case tables inline.


In [ ]:
res = steps.rapid_eval(client, RESOURCE)
raw = res.get('raw')
if raw is not None:
    raw.show()   # interactive summary + per-case scores
{k: v for k, v in res.items() if k != 'raw'}


## Phase 2b — Test-case / regression batch
📖 [evaluate-agents](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-agents)

Runs the per-agent regression suite against the deployed engine (this can take a few
minutes — it runs inference then a scored evaluation run).


In [ ]:
steps.testcase_eval(AGENT_ENGINE_ID, AGENT)


## Phase 2c — Simulated multi-turn evaluation
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Auto-generates multi-turn scenarios (starting prompt + hidden conversation plan) grounded
by `environment_context`, simulates the user, and scores with the multi-turn autoraters
(`MULTI_TURN_TASK_SUCCESS` / `_TOOL_USE_QUALITY` / `_TRAJECTORY_QUALITY`).


In [ ]:
steps.simulate(RESOURCE, AGENT, scenario_count=3, max_turns=4)


## Phase 2d — Environment simulation (resilience)
📖 [evaluate-simulated](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-simulated)

Intercept tool calls to inject mocked data and simulated failures (HTTP 503) — test how the
agent recovers, without touching production backends.


In [ ]:
steps.environment_simulation()


## Phase 2e — Offline evaluation over historical traces/sessions
📖 [evaluate-offline](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-offline)

Score **already-recorded** traces retroactively (no new inference). Reads gen_ai OTel
events from BigQuery, falling back to a bundled fixture.


In [ ]:
steps.offline_eval(client, AGENT)


## Phase 3 — Continuous evaluation with Online Monitors
📖 [evaluate-online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online)

Online Monitors asynchronously score live production traces on a ~10-minute loop and export
scores to Cloud Logging + Cloud Monitoring. Create/verify with
`python -m src.eval.setup_online_evaluators create`.


In [ ]:
steps.online_monitors(do_setup=False)


## Phase 4b — Optimize agent prompts (close the flywheel)
📖 [optimize-agent](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/optimize-agent)

Feature-detects the documented `client.optimizer.optimize(...)` and otherwise falls back to
the ADK GEPA optimizer to refine the root instruction against the eval suite.


In [ ]:
steps.optimize(client)


## Quality-drift alerts
📖 [quality-alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

Export a Cloud Monitoring alert policy on `aiplatform.googleapis.com/online_evaluator/scores`
and apply it with `gcloud monitoring policies create --policy-from-file=...`.


In [ ]:
steps.quality_alerts()


## Recap

You just ran the full Quality Flywheel: **Design → Execution → Scoring → Refinement**,
covering all nine Optimize → Evaluation doc pages. For the headless orchestrator + JSON
report, run:

```bash
uv run python -m src.eval.demo.full_eval_demo --agent-id $AGENT_ENGINE_ID --emit-json eval_outputs/demo/full_demo.json
```
